# TBP — Índice de Cuna Vacía · Laboratorio de auditoría y grupo de control (v2)

**Qué es este notebook:** un banco de pruebas para *auditar* la fórmula del grupo antes de calcular nada con datos de Argentina.

$$TBP(t)=\frac{N_t\,(1-\mu)\,\bar A\,\tau}{J_{t+65}\;E_r\;\rho}$$

**Cómo leerlo (de arriba hacia abajo):**

| Parte | Qué hace | ¿Usa datos reales? |
|---|---|---|
| 1 | Análisis dimensional (¿qué unidad tiene el TBP?) | No |
| 2 | Álgebra simbólica: qué se cancela, qué se descompone | No |
| 3 | **Grupo de control sintético**: un "mundo de juguete" donde conocemos la respuesta correcta y verificamos que el código y las interpretaciones coincidan | No (100 % ilustrativo) |
| 4 | Experimento de natalidad: ¿una cuna vacía baja el TBP? | No |
| 5 | Plantillas de datos reales + exigencia de fuente | Se cargan después |
| 6 | **Grupo de control con datos reales**: indicadores convencionales (tasa de dependencia, etc.) | Se cargan después |
| 7 | Checklist: qué buscar en las carpetas de Drive y qué NO se puede afirmar todavía | — |

> ⚠️ Todos los números de las Partes 1–4 son **ilustrativos** (mundo sintético). **No son datos de Argentina** y no deben citarse como tales.

## Hallazgos de la auditoría (resumen — el detalle se demuestra en las partes siguientes)

1. **Dimensionalmente la fórmula cierra** *solo si* $\rho$ es adimensional y está en la misma base salarial que $\tau$. La definición que lo permite es **$\rho$ = tasa de reemplazo = haber medio / salario medio**. Con eso el TBP es una **razón adimensional**: (aportes vitalicios de la cohorte) ÷ (prestaciones pendientes del stock de jubilados), ambos en unidades de salario.
2. **Problema de cohorte:** $J_{t+65}$ como *total de jubilados* mezcla decenas de cohortes distintas. Además, ese stock **ya contiene** a la cohorte $t$ (a los 65 años) y a las 35 cohortes anteriores.
3. **Longevidad contada dos veces:** $J_{t+65}$ (stock) ya crece cuando la gente vive más, y $E_r$ vuelve a multiplicar. En un mundo estacionario el TBP escala como $1/e_{65}^2$ (se demuestra en la Parte 3).
4. **Si $J$ se define como los sobrevivientes de la propia cohorte, $N_t$ se cancela** y el TBP deja de depender de la natalidad (Parte 2). La natalidad solo importa a través de su **tendencia relativa** a las cohortes previas (Parte 4), no de su nivel.
5. **La frase "cuántos años de jubilación compra la cohorte"** es correcta para la versión $TBP_C$ (en años) y para el balance individual $B$, pero **no** para $TBP_A$ tal cual está escrito.
6. **Ni TBP<1 ni TBP>1 significan "sistema insolvente / solvente"**: el umbral 1 es solo el punto donde los aportes vitalicios igualan la prestación pendiente en unidades de salario, sin descuento, sin crecimiento salarial, sin transferencias fiscales.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sympy as sp
from dataclasses import dataclass

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.width", 160)
try:
    display
except NameError:            # permite correr el notebook fuera de Jupyter/Colab
    display = print
print("Entorno listo.")

# PARTE 1 — Análisis dimensional

Cada variable se representa con sus exponentes de **personas**, **años** y **moneda**. Multiplicar suma exponentes, dividir resta. Si el resultado final tiene todos los exponentes en 0, el indicador es **adimensional** y las frases "TBP<1, =1, >1" tienen sentido.

In [ ]:
@dataclass(frozen=True)
class U:
    personas: int = 0
    años: int = 0
    moneda: int = 0
    def __mul__(s, o): return U(s.personas+o.personas, s.años+o.años, s.moneda+o.moneda)
    def __truediv__(s, o): return U(s.personas-o.personas, s.años-o.años, s.moneda-o.moneda)
    def es_adimensional(s): return s.personas == s.años == s.moneda == 0
    def __str__(s):
        partes = [f"{n}^{e}" for n, e in [("personas", s.personas), ("años", s.años), ("moneda", s.moneda)] if e]
        return " · ".join(partes) if partes else "adimensional"

PROP = U()  # proporción
unidades = {
    "N_t (nacimientos)":           U(personas=1),
    "1-mu (supervivencia)":        PROP,
    "A_barra (años de aporte)":    U(años=1),
    "tau (tasa de aporte)":        PROP,
    "J_t+65 (jubilados)":          U(personas=1),
    "E_r (años restantes)":        U(años=1),
}

def dim_tbp(rho_unidad):
    num = unidades["N_t (nacimientos)"] * unidades["1-mu (supervivencia)"] * unidades["A_barra (años de aporte)"] * unidades["tau (tasa de aporte)"]
    den = unidades["J_t+65 (jubilados)"] * unidades["E_r (años restantes)"] * rho_unidad
    return num, den, num/den

casos_rho = {
    "rho = tasa de reemplazo (haber/salario)  → adimensional": PROP,
    "rho = haber medio en $ por año            → tiene moneda": U(moneda=1, años=-1),
    "rho = gasto previsional / PBI             → adimensional pero base distinta a tau": PROP,
}
filas = []
for nombre, u in casos_rho.items():
    num, den, tot = dim_tbp(u)
    filas.append({"definición de rho": nombre, "numerador": str(num), "denominador": str(den), "TBP": str(tot),
                  "¿adimensional?": tot.es_adimensional()})
display(pd.DataFrame(filas))
print("\nLectura: con rho = tasa de reemplazo, numerador y denominador son 'personas·años' y el TBP es adimensional.")
print("Con rho en $, el TBP queda en 'moneda^-1 · años^..' y NO permite hablar de TBP<1 / >1.")
print("El tercer caso es adimensional en unidades, pero rho (%PBI) y tau (%salario) tienen bases distintas: cierra la unidad, no la economía.")

# PARTE 2 — Álgebra: qué se cancela y cómo se descompone

Tres versiones que vamos a comparar (todas conservan los mismos términos, cambian qué es $J$ y si aparece $E_r$):

* **$TBP_A$ (fórmula original):** $\dfrac{N(1-\mu)\bar A\tau}{J\,E_r\,\rho}$ con $J$ = stock de jubilados en $t+65$.
* **$TBP_B$ (cohorte pura):** $J := N(1-\mu)$ ⇒ $\dfrac{\bar A\tau}{E_r\rho}$ — balance actuarial individual.
* **$TBP_C$ (en años):** $\dfrac{N(1-\mu)\bar A\tau}{J\,\rho}$ — *años de prestación por jubilado que el aporte vitalicio de la cohorte podría financiar*. Se compara contra $E_r$: $TBP_A = TBP_C / E_r$.

In [ ]:
N, mu, A, tau, J, E, rho = sp.symbols("N mu A tau J E rho", positive=True)

TBP_A = N*(1-mu)*A*tau / (J*E*rho)
TBP_B = TBP_A.subs(J, N*(1-mu))          # J = sobrevivientes de la propia cohorte
TBP_C = N*(1-mu)*A*tau / (J*rho)

D = N*(1-mu)/J                           # factor demográfico: cohorte que llega a 65 / stock de jubilados
B = A*tau/(E*rho)                        # balance actuarial individual

print("TBP_B (J = N(1-mu)) =", sp.simplify(TBP_B), "   ← N desapareció")
print("¿TBP_A == D × B ?      ", sp.simplify(TBP_A - D*B) == 0)
print("¿TBP_A == TBP_C / E ?  ", sp.simplify(TBP_A - TBP_C/E) == 0)

# Elasticidades (d ln TBP / d ln x)
print("\nElasticidades de TBP_A:")
for v in [N, mu, A, tau, J, E, rho]:
    print(f"  {str(v):>3}: ", sp.simplify(sp.diff(sp.log(TBP_A), v)*v))

**Lectura:**

* $TBP_A = D \times B$: un **factor demográfico** $D$ (¿cuántos llegan a 65 relativo al stock de jubilados?) por un **balance individual** $B$ (años aportados × tasa ÷ años cobrados × reemplazo).
* La natalidad vive **solo en $D$**. Y $D$ depende del *cociente* entre la cohorte y el stock, no del nivel de $N$.
* La elasticidad de $\mu$ es $-\mu/(1-\mu)$ (no ±1 como las demás), por eso conviene reportar $1-\mu$.

# PARTE 3 — Grupo de control sintético ("mundo de juguete")

**Idea:** construimos un mundo donde *sabemos analíticamente* cuánto debería dar cada versión. Si el código y las interpretaciones coinciden, la implementación es confiable. Si no, hay un error antes de tocar datos reales.

**Mundo estacionario de De Moivre** (todo ilustrativo): límite de edad $\omega=90$, supervivencia $l(a)=1-a/\omega$, nacimientos constantes. Resultados exactos:

* supervivencia a 65: $S_{65}=1-65/\omega$
* stock de 65+ por nacimiento: $(\omega-65)^2/(2\omega)$
* esperanza de vida a 65: $e_{65}=(\omega-65)/2$

In [ ]:
OMEGA = 90.0
def l(a, omega=OMEGA):
    a = np.asarray(a, dtype=float)
    return np.clip(1 - a/omega, 0, 1)

def tbp_familia(N, mu, A, tau, J, E, rho):
    a = N*(1-mu)*A*tau / (J*E*rho)
    return {"TBP_A": a,
            "TBP_B": A*tau/(E*rho),
            "TBP_C": N*(1-mu)*A*tau / (J*rho),
            "D": N*(1-mu)/J,
            "B": A*tau/(E*rho)}

# Parámetros ILUSTRATIVOS (no son de Argentina)
par = dict(A=30.0, tau=0.20, rho=0.60)

dt = 0.01
edades = np.arange(0, OMEGA, dt)
S65 = float(l(65))
stock_por_nac = float(np.sum(l(edades[edades >= 65])) * dt)     # integral numérica
e65 = stock_por_nac / S65

resultados = []
def check(nombre, obtenido, esperado, tol=1e-3):
    ok = abs(obtenido - esperado) <= tol*max(1, abs(esperado))
    resultados.append({"test": nombre, "obtenido": obtenido, "esperado": esperado, "OK": "✅" if ok else "❌"})

# T1: fórmulas cerradas del mundo estacionario
check("S65 = 1-65/omega", S65, 1-65/OMEGA)
check("stock 65+ por nacimiento = (omega-65)^2/(2 omega)", stock_por_nac, (OMEGA-65)**2/(2*OMEGA))
check("e65 = (omega-65)/2", e65, (OMEGA-65)/2)

# T2: en estado estacionario, J = N*S65*e65 (el stock ES los sobrevivientes × años que les quedan)
N0 = 1000.0
mu0 = 1 - S65
J0 = N0*stock_por_nac
f = tbp_familia(N0, mu0, par["A"], par["tau"], J0, e65, par["rho"])
check("J = N*S65*e65", J0, N0*S65*e65)

# T3: cierre analítico de cada versión
check("TBP_B = A*tau/(e65*rho)", f["TBP_B"], par["A"]*par["tau"]/(e65*par["rho"]))
check("TBP_A = A*tau/(e65^2*rho)  ← longevidad entra al cuadrado", f["TBP_A"], par["A"]*par["tau"]/(e65**2*par["rho"]))
check("TBP_A = D * B", f["TBP_A"], f["D"]*f["B"])
check("TBP_A = TBP_C / E_r", f["TBP_A"], f["TBP_C"]/e65)
check("D = 1/e65 en estado estacionario", f["D"], 1/e65)

# T4: el NIVEL de nacimientos no debe importar (dos mundos, uno con el doble de nacimientos)
f2 = tbp_familia(2*N0, mu0, par["A"], par["tau"], 2*J0, e65, par["rho"])
check("Duplicar N (y J proporcional) no cambia TBP_A", f2["TBP_A"], f["TBP_A"])

# T5: TBP_B no depende de N aunque cambie J arbitrariamente
check("TBP_B no depende de N", tbp_familia(5*N0, mu0, par["A"], par["tau"], J0, e65, par["rho"])["TBP_B"], f["TBP_B"])

# T6: signos (ceteris paribus)
base = dict(N=N0, mu=mu0, A=par["A"], tau=par["tau"], J=J0, E=e65, rho=par["rho"])
def tbp_a(**kw): return tbp_familia(**{**base, **kw})["TBP_A"]
signos = {"N ↑": tbp_a(N=1.1*N0) > f["TBP_A"], "1-mu ↑": tbp_a(mu=mu0-0.05) > f["TBP_A"],
          "A ↑": tbp_a(A=35) > f["TBP_A"], "tau ↑": tbp_a(tau=0.25) > f["TBP_A"],
          "J ↑ → TBP ↓": tbp_a(J=1.1*J0) < f["TBP_A"], "E_r ↑ → TBP ↓": tbp_a(E=e65+2) < f["TBP_A"],
          "rho ↑ → TBP ↓": tbp_a(rho=0.7) < f["TBP_A"]}
for k, v in signos.items():
    resultados.append({"test": f"signo: {k}", "obtenido": float(v), "esperado": 1.0, "OK": "✅" if v else "❌"})

res_df = pd.DataFrame(resultados)
display(res_df)
assert (res_df["OK"] == "✅").all(), "Hay tests de control que fallan: revisar implementación."
print(f"\n{len(res_df)} tests del grupo de control sintético: TODOS OK.")
print(f"Ilustración (mundo sintético): TBP_A={f['TBP_A']:.3f}, TBP_B={f['TBP_B']:.3f}, TBP_C={f['TBP_C']:.2f} años vs e65={e65:.1f}")

**Qué demuestra esta parte**

* El test `TBP_A = A·τ/(e65²·ρ)` verifica el hallazgo #3: en la fórmula original **la longevidad entra dos veces** (una vía $J$, otra vía $E_r$). No es un error de programación; es una propiedad de la fórmula que hay que **justificar o corregir** (usar $TBP_B$ o $TBP_C$).
* El test de "duplicar $N$" muestra que **el nivel de nacimientos, por sí solo, no mueve el TBP** si $J$ se calcula de manera consistente.

# PARTE 4 — Experimento de natalidad: ¿una cuna vacía baja el TBP?

Tres escenarios en el mundo sintético, con **todo lo demás constante**. $J_{t+65}=\sum_{a\ge 65} N_{t+65-a}\,l(a)$: el stock de jubilados se construye con los nacimientos de las cohortes $t-35,\dots,t$.

1. **Nivel:** todas las cohortes tienen la mitad de nacimientos.
2. **Shock puntual:** solo la cohorte $t$ cae 20 %.
3. **Tendencia sostenida:** los nacimientos caen $g$ % por año durante las cohortes previas (la cohorte $t$ queda igual, las anteriores eran más grandes).

In [ ]:
def stock_J(nac_por_cohorte, t_idx):
    # nac_por_cohorte[k] = nacimientos de la cohorte t-k  (k=0 es la cohorte t). J = sum_{a>=65} N_{t+65-a} l(a)
    edades_int = np.arange(65, int(OMEGA))
    return sum(nac_por_cohorte[a-65] * float(l(a)) for a in edades_int)

def tbp_desde_cohortes(nac, A=par["A"], tau=par["tau"], rho=par["rho"]):
    Nt = nac[0]
    Jt = stock_J(nac, 0)
    S = float(l(65))
    # esperanza de vida a 65 (discreta, consistente con la grilla anual)
    e = sum(float(l(a)) for a in range(65, int(OMEGA))) / S
    return tbp_familia(Nt, 1-S, A, tau, Jt, e, rho)

n_coh = int(OMEGA) - 65
base_nac = np.full(n_coh, 1000.0)
ref = tbp_desde_cohortes(base_nac)

filas = [{"escenario": "Referencia (nacimientos constantes)", "N_t": 1000, "TBP_A": ref["TBP_A"], "D": ref["D"], "TBP_B": ref["TBP_B"], "TBP_C": ref["TBP_C"]}]

r = tbp_desde_cohortes(base_nac*0.5)
filas.append({"escenario": "1) Nivel: TODAS las cohortes a la mitad", "N_t": 500, **{k: r[k] for k in ["TBP_A","D","TBP_B","TBP_C"]}})

pulso = base_nac.copy(); pulso[0] *= 0.8
r = tbp_desde_cohortes(pulso)
filas.append({"escenario": "2) Shock: solo cohorte t cae 20%", "N_t": 800, **{k: r[k] for k in ["TBP_A","D","TBP_B","TBP_C"]}})

for g in [-0.01, -0.02, -0.03]:
    # cohorte t (k=0) = 1000; la cohorte k años antes era 1000*(1+g)^(-k) (más grande si g<0)
    nac = 1000.0 * (1+g)**(-np.arange(n_coh))
    r = tbp_desde_cohortes(nac)
    filas.append({"escenario": f"3) Tendencia: nacimientos {g*100:.0f}%/año", "N_t": 1000, **{k: r[k] for k in ["TBP_A","D","TBP_B","TBP_C"]}})

exp = pd.DataFrame(filas)
exp["TBP_A vs ref"] = exp["TBP_A"]/ref["TBP_A"] - 1
display(exp)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
gs = np.linspace(0, -0.04, 21)
vals = [tbp_desde_cohortes(1000.0*(1+g)**(-np.arange(n_coh)))["TBP_A"] for g in gs]
ax[0].plot(gs*100, vals, marker="o"); ax[0].set_xlabel("tendencia de nacimientos (% anual)")
ax[0].set_ylabel("TBP_A (sintético)"); ax[0].set_title("Tendencia sostenida de natalidad → TBP_A"); ax[0].grid(alpha=.3)

pulsos = np.linspace(0, -0.5, 21)
vals2 = []
for p in pulsos:
    nac = base_nac.copy(); nac[0] *= (1+p); vals2.append(tbp_desde_cohortes(nac)["TBP_A"])
ax[1].plot(pulsos*100, vals2, marker="o", color="tab:red"); ax[1].set_xlabel("caída puntual de la cohorte t (%)")
ax[1].set_title("Shock puntual en N_t → TBP_A"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

**Lectura (mundo sintético):**

* El **nivel** de nacimientos no altera el TBP (escenario 1) → la hipótesis "menos nacimientos ⇒ menor TBP" **no es testeable en niveles**.
* Un **shock puntual** en $N_t$ baja el TBP casi proporcionalmente (−18,75 % ante −20 %; escenario 2), porque una sola cohorte pesa poco en el stock $J$.
* Una **tendencia sostenida** de caída de nacimientos también baja el TBP (escenario 3), pero **por otra vía**: $N_t$ se vuelve chico *relativo* a las cohortes previas que componen $J$ (cae el factor demográfico $D$). La hipótesis debe formularse en términos de **tendencia relativa**: *"cuando $N_t$ cae respecto de las cohortes que integran el stock de jubilados en $t+65$, cae el TBP"*.
* Cuidado: en datos reales, $J_{t+65}$ para cohortes recientes es una **proyección** que ya depende de los nacimientos observados. Hay riesgo de circularidad que hay que documentar.

# PARTE 5 — Datos reales de Argentina: plantillas con trazabilidad

**Regla de oro: si un dato no tiene fuente, no entra.** Por eso el notebook exige dos tablas:

1. `tbp_datos.csv`: una fila por cohorte $t$.
2. `tbp_fuentes.csv`: una fila por variable con fuente, serie, unidad, período, enlace y tipo (*observado / proyectado / proxy*).

No se cargan valores inventados. Las plantillas se generan vacías.

In [ ]:
import os
REQUIRED = ["year", "births", "mu", "avg_contribution_years", "contribution_rate",
            "retirees_t65", "life_expectancy_65", "rho", "data_type"]

VAR_DEF = pd.DataFrame([
    ["year", "Año de nacimiento de la cohorte t", "año", "—"],
    ["births", "N_t: nacimientos en el año t", "personas", "INDEC (estadísticas vitales) — a verificar serie exacta"],
    ["mu", "1-mu = supervivencia de la cohorte hasta 65 (l65/l0) — propuesta a validar", "proporción [0,1]", "Tablas de mortalidad INDEC / proyecciones"],
    ["avg_contribution_years", "A_barra: años de aporte esperados por persona de la cohorte (incluye ceros)", "años", "SIPA/ANSES u otra — a definir"],
    ["contribution_rate", "tau: tasa de aporte (personal + patronal jubilatoria) — a definir y justificar", "proporción [0,1]", "Normativa vigente (ANSES/AFIP-ARCA) — a verificar"],
    ["retirees_t65", "J_{t+65}: stock de 65+ (o beneficiarios) en t+65 — a definir (ver Alt. A/B/C)", "personas", "Proyecciones de población — a verificar horizonte"],
    ["life_expectancy_65", "E_r: esperanza de vida restante a los 65", "años", "INDEC / proyecciones — a definir sexo/ponderación"],
    ["rho", "rho: tasa de reemplazo = haber medio / salario medio — propuesta a validar", "proporción (>0)", "ANSES/SIPA + RIPTE — a verificar"],
    ["data_type", "observado / proyectado / proxy", "texto", "—"],
], columns=["variable", "definición", "unidad", "fuente candidata (SIN verificar)"])
display(VAR_DEF)

FUENTES_COLS = ["variable", "fuente", "serie_exacta", "unidad", "periodo", "url", "tipo", "notas"]

def generar_plantillas(carpeta="."):
    pd.DataFrame(columns=REQUIRED).to_csv(os.path.join(carpeta, "tbp_datos.csv"), index=False)
    pd.DataFrame({"variable": [v for v in REQUIRED if v not in ("year", "data_type")],
                  **{c: "" for c in FUENTES_COLS[1:]}}).to_csv(os.path.join(carpeta, "tbp_fuentes.csv"), index=False)
    pd.DataFrame(columns=["year", "dependency_ratio_65plus", "support_ratio", "active_passive_ratio", "fuente", "url"]).to_csv(
        os.path.join(carpeta, "tbp_control.csv"), index=False)
    print("Plantillas creadas en:", os.path.abspath(carpeta))

# generar_plantillas("/content")   # ← descomentar en Colab la primera vez, completar los CSV y subirlos

def leer_csv(path, columnas):
    return pd.read_csv(path) if os.path.exists(path) else pd.DataFrame(columns=columnas)

df = leer_csv("/content/tbp_datos.csv", REQUIRED)
fuentes = leer_csv("/content/tbp_fuentes.csv", FUENTES_COLS)
control = leer_csv("/content/tbp_control.csv", ["year", "dependency_ratio_65plus", "support_ratio", "active_passive_ratio", "fuente", "url"])
print(f"Filas TBP: {len(df)} | filas de fuentes: {len(fuentes)} | filas de control: {len(control)}")

In [ ]:
def validar(df, fuentes):
    errores, avisos = [], []
    faltan = [c for c in REQUIRED if c not in df.columns]
    if faltan:
        return [f"Faltan columnas: {faltan}"], avisos
    if len(df) == 0:
        return [], ["Todavía no hay datos cargados (esperado en esta etapa)."]
    if df["year"].duplicated().any(): errores.append("Años de cohorte duplicados.")
    if df[REQUIRED].isna().any().any(): avisos.append("Hay valores faltantes.")
    for c in ["births", "avg_contribution_years", "retirees_t65", "life_expectancy_65"]:
        if (df[c] < 0).any(): errores.append(f"{c}: valores negativos.")
    for c in ["mu", "contribution_rate"]:
        if ((df[c] < 0) | (df[c] > 1)).any(): errores.append(f"{c}: fuera de [0,1].")
    if (df["rho"] <= 0).any(): errores.append("rho debe ser > 0.")
    if (df["rho"] > 1.5).any(): avisos.append("rho > 1.5: ¿está en % en vez de proporción? ¿es un monto en $?")
    if (df["contribution_rate"] > 0.5).any(): avisos.append("tau > 0.5: revisar unidades (¿% en vez de proporción?).")
    if not set(df["data_type"].dropna()) <= {"observado", "proyectado", "proxy"}:
        errores.append("data_type solo puede ser observado / proyectado / proxy.")
    # Trazabilidad: cada variable necesita fuente y URL
    for v in [c for c in REQUIRED if c not in ("year", "data_type")]:
        fila = fuentes[fuentes["variable"] == v]
        if fila.empty or fila[["fuente", "url"]].replace("", np.nan).isna().any(axis=None):
            errores.append(f"Variable '{v}' sin fuente/URL en tbp_fuentes.csv → no puede entrar como dato oficial.")
    return errores, avisos

errores, avisos = validar(df, fuentes)
for e in errores: print("❌", e)
for a in avisos:  print("⚠️", a)
if not errores and len(df): print("✅ Validación estructural y de trazabilidad OK.")

In [ ]:
def calcular_tbp(df):
    d = df.copy()
    d["numerador"] = d["births"]*(1-d["mu"])*d["avg_contribution_years"]*d["contribution_rate"]
    d["TBP_A"] = d["numerador"]/(d["retirees_t65"]*d["life_expectancy_65"]*d["rho"])
    d["TBP_B"] = d["avg_contribution_years"]*d["contribution_rate"]/(d["life_expectancy_65"]*d["rho"])
    d["TBP_C_años"] = d["numerador"]/(d["retirees_t65"]*d["rho"])
    d["D_demografico"] = d["births"]*(1-d["mu"])/d["retirees_t65"]
    # Chequeo interno: dos caminos de cálculo independientes deben coincidir
    assert np.allclose(d["TBP_A"], d["D_demografico"]*d["TBP_B"]), "TBP_A ≠ D×B: error de implementación"
    assert np.allclose(d["TBP_A"], d["TBP_C_años"]/d["life_expectancy_65"]), "TBP_A ≠ TBP_C/E_r: error de implementación"
    return d

result = pd.DataFrame()
if len(df) and not errores:
    result = calcular_tbp(df)
    display(result[["year", "data_type", "TBP_A", "TBP_B", "TBP_C_años", "D_demografico"]])
else:
    print("Esperando datos válidos y con fuente para calcular.")

# PARTE 6 — Grupo de control con datos reales

Cuando haya datos, comparamos el TBP con **indicadores convencionales independientes**. No es una "validación" del TBP: es una **prueba de razonabilidad**.

| Indicador de control | Signo esperado vs TBP |
|---|---|
| Tasa de dependencia 65+ / población en edad activa | **negativo** (más carga → menos TBP) |
| Relación de sostenimiento (activos / 65+) | **positivo** |
| Relación activos/pasivos previsional (SIPA) | **positivo** |

* Si los signos coinciden → coherencia mínima.
* Si divergen → **no es automáticamente un error**: investigamos si se debe a la definición de $J$, a la longevidad, a $\rho$ o a que miden cosas distintas.
* Los controles deben venir de **fuentes distintas** a las variables del TBP (para no medir lo mismo dos veces).

In [ ]:
ESPERADO = {"dependency_ratio_65plus": -1, "support_ratio": +1, "active_passive_ratio": +1}

comparison = pd.DataFrame()
if len(result) and len(control):
    comparison = result.merge(control.drop(columns=["fuente", "url"], errors="ignore"), on="year", how="inner")

if len(comparison) >= 5:
    filas = []
    for ctrl, signo in ESPERADO.items():
        if ctrl in comparison and comparison[ctrl].notna().sum() >= 5:
            for v in ["TBP_A", "TBP_C_años", "TBP_B"]:
                rho_s = comparison[[v, ctrl]].corr(method="spearman").iloc[0, 1]
                filas.append({"control": ctrl, "versión TBP": v, "Spearman": rho_s,
                              "signo esperado": "+" if signo > 0 else "−",
                              "coherente": "✅" if np.sign(rho_s) == signo else "⚠️ investigar"})
    display(pd.DataFrame(filas))
    print("Recordatorio: correlación ≠ validación ni causalidad. Las divergencias se explican, no se esconden.")
else:
    print("Se necesitan ≥5 años con TBP y con controles para comparar. (Etapa pendiente: cargar datos.)")

In [ ]:
if len(result):
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.plot(result["year"], result["TBP_A"], marker="o", label="TBP_A")
    ax.plot(result["year"], result["TBP_B"], marker="s", label="TBP_B (cohorte pura)")
    ax.axhline(1, ls="--", color="gray", label="razón = 1 (no es frontera fiscal)")
    ax.set_xlabel("Cohorte t"); ax.set_ylabel("razón adimensional"); ax.legend(); ax.grid(alpha=.3)
    ax.set_title("TBP por cohorte"); plt.show()
else:
    print("Sin datos para graficar.")

# PARTE 7 — Checklist para las carpetas de Drive y límites de lo que se puede afirmar

## Qué buscar en los documentos del grupo (Etapa 2 y 4)
- [ ] ¿Cómo definió el grupo $\mu$? (¿mortalidad hasta edad laboral, hasta 65, otra?)
- [ ] ¿Qué es $J_{t+65}$ en sus fuentes: población 65+, beneficiarios, jubilados SIPA?
- [ ] ¿Qué $\tau$ usaron (personal 11 %, patronal, ambas)? *(a verificar contra normativa vigente)*
- [ ] ¿Qué es $\rho$ en sus documentos? ¿Haber medio? ¿Tasa de reemplazo? ¿Costo/PBI?
- [ ] Serie de años de aporte $\bar A$ y su fuente (¿solo aportantes o toda la cohorte?)
- [ ] Series de INDEC: nacimientos, tablas de mortalidad, proyecciones (¿hasta qué año llegan?).
- [ ] Series ANSES/SIPA: beneficiarios, haber medio, aportantes.
- [ ] Series de control: población por grandes grupos de edad, relación activos/pasivos.

## Qué NO podemos afirmar todavía
- Que el TBP sea *novedoso* (falta el estado del arte).
- Que $TBP<1$ implique déficit, insolvencia o "quiebra" del sistema.
- Que la caída de natalidad *cause* una caída del TBP (depende de la tendencia relativa y de cómo se defina $J$).
- Que existan proyecciones oficiales de población a $t+65$ para las cohortes recientes (el horizonte puede no alcanzar; verificar).
- Cualquier valor numérico para Argentina: **ninguno está cargado**.

## Qué NO mide el TBP (aun bien construido)
Financiamiento fiscal total (rentas generales, deuda, ANSES-FGS), crecimiento salarial y productividad, informalidad *vs* moratorias, migración, cambios legislativos futuros, heterogeneidad por sexo y por nivel de ingreso.

## Fe de erratas (02-10-2026)

La plantilla de la Parte 5 cita a **INDEC (estadísticas vitales)** como fuente de `births`. La serie de nacimientos que se usó en el cálculo real es la del **DEIS (Ministerio de Salud)**: *Serie histórica de nacimientos ocurridos en Argentina por jurisdicción*, 1914-2024. Ver `TBP_Argentina_Series_Reales.ipynb`. El texto original se deja arriba tal como se presentó.